<a href="https://colab.research.google.com/github/syakesaba/jupyter-notebooks/blob/main/github_copilot_sdk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Github Copilot SDK Python Sample Code
=====


Ref: https://github.com/github/copilot-sdk/blob/main/docs/features/index.md

# Initialize Python

In [21]:
!uv pip install --system github-copilot-sdk nest_asyncio pydantic httpx
import nest_asyncio
nest_asyncio.apply() # Google Colab自体がasyncio配下で動いているのでネストさせる。
from google.colab import userdata
GH_TOKEN=userdata.get("GH_TOKEN") # Secrets（🔑）からGH_TOKENをNotebook access可能にする
# GH_TOKEN=`gh auth token`

Using Python 3.12.13 environment at: /usr
Checked 4 packages in 156ms


# Github Copilot Client

In [22]:
import asyncio
from copilot import CopilotClient, SubprocessConfig

client = CopilotClient(
    SubprocessConfig(
        use_logged_in_user=False,
        github_token=GH_TOKEN,
        log_level="debug",
    )
)

asyncio.run(client.start())

# Prompting


## 最も基本的なプロンプト

In [23]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1",
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("Answer what 2+2 is."))

2 + 2 equals 4.


## システムプロンプト

In [24]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler, SystemMessageReplaceConfig

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1",
        system_message=SystemMessageReplaceConfig(content="あなたは真逆のことを言うエージェントです")
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("多くのカラスは黒いですか？"))

いいえ、多くのカラスは黒くありません。


# Tool

In [25]:
import asyncio

from copilot import CopilotSession, define_tool
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler, SystemMessageReplaceConfig

from pydantic import BaseModel, Field, StringConstraints

class Loto6Params(BaseModel):
    number: str = Field(description="Number for LOTO6", pattern=r'^\d{6}$')

# name must be: ^[a-zA-Z0-9_\\.-]+$
@define_tool(name="Loto6Tool", description="Loto6 check if win or not!", skip_permission=True)
async def _loto6(params: Loto6Params) -> str:
    return "WOW you got $10,000 !" if "123456" == params.number else "Sorry you got nothing!"

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1",
        tools=[_loto6,],
        system_message=SystemMessageReplaceConfig(content="Use `Loto6Tool` if you receive 6-digits numbers and answer what you got.")
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("My number is: 123456"))


Congratulations! Your number 123456 is a winner—you've got $10,000!


# Image Input

---



In [26]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler

from pydantic import AnyUrl
import httpx
import base64

async def run(prompt: str, url: AnyUrl):
    async with httpx.AsyncClient() as http_client:
        image_response = await http_client.get(url)
        bin_image = image_response.content
        image_type = image_response.headers['content-type']
        b64_image = base64.b64encode(bin_image).decode('utf-8')
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1"
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(
        prompt,
        attachments=[
            {
                "type": "blob",
                "data": b64_image,
                "mimeType": image_type,
            }
        ]
    )
    await done.wait()
    await session.disconnect()

asyncio.run(run("これ何？", "http://www.sakado-jigenji.jp/images/k_logo.png"))

この画像は「慈眼寺（じげんじ）」というお寺のロゴやタイトル部分です。

上部には「真言宗 智山派 由城山」と書かれており、これは慈眼寺が「真言宗智山派」に属し、山号が「由城山」であることを示しています。左側のマークは寺院の紋章（家紋のようなもの）です。

要約：
- 慈眼寺（じげんじ）という真言宗智山派のお寺のロゴ・タイトル画像です。


# Custom Agents & Sub-Agent Orchestration
(Multi-Agent-System)

In [27]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler, CustomAgentConfig, SystemMessageReplaceConfig

agent1 = CustomAgentConfig(
    name="Agent1",
    display_name="Agent1",
    description="Agent1",
    prompt="ケーキの見た目についてのみのプロフェッショナルとして回答します。",
    infer=True
)

agent2 = CustomAgentConfig(
    name="Agent2",
    display_name="Agent2",
    description="Agent2",
    prompt="ケーキの味についてのみのプロフェッショナルとして回答します。",
    infer=True
)

agent3 = CustomAgentConfig(
    name="Agent3",
    display_name="Agent3",
    description="Agent3",
    prompt="最高のケーキについて、Agent1とAgent2に確認した上で総合的に回答します。",
    infer=False
)

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1",
        custom_agents=[agent1, agent2, agent3],
        system_message=SystemMessageReplaceConfig(content="Jemmy's TeaHouse"),
        agent="Agent3"
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("私は太郎です。ケーキについて知りたい。"))


Agent1とAgent2に「最高のケーキ」について確認しています。回答が揃い次第、総合的な情報をお伝えしますので、少々お待ちください。
ケーキの見た目に特化したプロフェッショナルの視点から「最高のケーキ」をご紹介します。

私が考える最高のケーキは、「シンプルで洗練されたデザインのショートケーキ」です。

**理由・おすすめポイント：**

1. **清潔感と上品さ**
   - 真っ白な生クリームに、鮮やかな赤いイチゴが映えるコントラストは、誰が見ても美しく、清潔感があります。
   - 余計な装飾をせず、素材の美しさを最大限に引き出すことで、上品な印象を与えます。

2. **バランスの良いレイアウト**
   - イチゴの配置やクリームの絞り方が均一で整っていると、全体のバランスが良く、見ていて心地よいです。
   - ケーキの断面も美しく、スポンジとクリーム、イチゴの層がはっきり見えると、カットしたときの感動も大きいです。

3. **季節感や特別感の演出**
   - 季節のフルーツやエディブルフラワーをアクセントに加えることで、特別な日や季節感を演出できます。
   - シンプルな中にも、ちょっとした工夫で個性を出せるのも魅力です。

4. **写真映え**
   - ショートケーキは、どの角度から撮影しても美しく、SNS映えも抜群です。
   - シンプルな背景やお皿に乗せることで、よりケーキの美しさが引き立ちます。

**まとめ：**
最高のケーキは、見た瞬間に「食べてみたい！」と思わせる美しさと、シンプルな中にこだわりが感じられるデザインです。ショートケーキはその代表例であり、誰にでも愛される見た目の良さがあります。

他にもご希望のスタイルやテーマがあれば、ぜひ教えてください！
私がプロフェッショナルとして考える「最高のケーキ」は、「ショートケーキ（いちごの生クリームケーキ）」です。

理由・おすすめポイント：

1. **バランスの良い味わい**  
   ふわふわのスポンジ、軽やかな生クリーム、甘酸っぱいいちごの組み合わせは、甘さ・酸味・コクのバランスが絶妙です。どの層も主張しすぎず、全体として調和のとれた味わいになります。

2. **食感のコントラスト**  
   しっとりとしたスポンジと、なめらかなクリーム、そしてジューシーないち

# MCP Servers

In [28]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler, SystemMessageReplaceConfig, MCPStdioServerConfig

server_time = MCPStdioServerConfig(
    type="stdio",
    env={},
    cwd="",
    command="uvx",
    args=["mcp-server-time", "--local-timezone=Asia/Tokyo"],
    tools="*",
)

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1",
        mcp_servers=[server_time,],
        system_message=SystemMessageReplaceConfig(content="サーバ時刻を伝えます")
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("今何時？"))


現在の日本時間（JST）は2026年5月7日（木）7時14分です。


# Skills

In [29]:
!npx -y skills add -y vercel-labs/agent-skills 1>/dev/null
!pwd
!ls -la .agents/skills/

/content
total 36
drwxr-xr-x 9 root root 4096 May  6 22:14 .
drwxr-xr-x 3 root root 4096 May  6 21:32 ..
drwxr-xr-x 3 root root 4096 May  6 22:14 deploy-to-vercel
drwxr-xr-x 2 root root 4096 May  6 22:14 vercel-cli-with-tokens
drwxr-xr-x 3 root root 4096 May  6 22:14 vercel-composition-patterns
drwxr-xr-x 3 root root 4096 May  6 22:14 vercel-react-best-practices
drwxr-xr-x 3 root root 4096 May  6 22:14 vercel-react-native-skills
drwxr-xr-x 3 root root 4096 May  6 22:14 vercel-react-view-transitions
drwxr-xr-x 2 root root 4096 May  6 22:14 web-design-guidelines


In [30]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1",
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("Answer what 2+2 is."))

2 + 2 equals 4.
